# 12 — CNN rung 4, experiment 4: class_weight="balanced"

**Decision this feeds** (`README.md`): flagged since the EDA as "an
experiment to validate, not a default" — never actually tried. Weights
`BCEWithLogitsLoss`'s `pos_weight` at `n_negative / n_positive`
(`evaluate.compute_pos_weight`, TDD'd 2026-09-09), computed per fold
from the actual inner-train labels.

**Expectation going in, stated plainly**: the rung 2/3 design spec
already argued explicitly *against* this as a default — log loss is a
proper scoring rule, and `pos_weight` distorts the loss surface toward
predicting the minority class more, which risks hurting calibration
(the exact thing log loss penalizes). Class balance here is also mild
(54.8%/45.2%, `config.BASE_RATE`) — weaker case for needing it than a
severely imbalanced problem. Testing it anyway because it was on the
original list, but a negative result here would not be surprising.

**Gate**: same as experiments 1-3 — nested-CV + paired bootstrap against
the **current validated CNN** (rung 3, `README.md` 2026-09-09:
mean=0.4520, sd=0.0109). No hyperparameter search (pos_weight is
computed, not tuned) — fold-0 sanity check, then the full 5×5 gate.

**Data handling**: this notebook loads real `.nii.gz` volumes and
row-level labels throughout, so per the AI-assistant data rule
(`README.md`) it is **[RUN ME]** — run it yourself, share back only the
printed aggregate numbers, never any per-row output.

In [ ]:
# [RUN ME] -- loads real pixel data + row-level labels. Reuses the
# shared on-disk volume cache (this experiment doesn't touch
# preprocessing, so cell 08's "reused" result should hold).
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import torch

import cache
import config
import dataset
import evaluate
import model
import train as train_mod

labels_df = pd.read_csv(config.TRAIN_LABELS_PATH)
family_df = pd.read_csv(config.DATA_PROCESSED / "baseline_features.csv")[
    [config.UID_COLUMN, "inplane_family"]
]
labeled_df = labels_df.merge(family_df, on=config.UID_COLUMN, how="inner").reset_index(drop=True)

uids = labeled_df[config.UID_COLUMN].tolist()
labels = labeled_df[config.TARGET_COLUMN].tolist()
families = labeled_df["inplane_family"].tolist()

config_fingerprint = {
    "TARGET_SPACING": config.TARGET_SPACING,
    "CROP_SIZE_MM": config.CROP_SIZE_MM,
    "CROP_CENTER_MM": config.CROP_CENTER_MM,
    "TARGET_SHAPE": config.TARGET_SHAPE,
    "BACKGROUND_PERCENTILE": config.BACKGROUND_PERCENTILE,
    "BACKGROUND_MAX_FRACTION": config.BACKGROUND_MAX_FRACTION,
}

cache_start = time.time()
volume_cache = cache.CachedVolumeStore(
    uids, cache_dir=config.DATA_PROCESSED / "volume_cache",
    config_fingerprint=config_fingerprint,
)
print(f"cache {'reused' if volume_cache.was_reused else 'rebuilt'} in "
      f"{time.time() - cache_start:.1f}s for {len(uids)} volumes")

In [ ]:
# [RUN ME] (no data access itself). Same helper as notebooks 06/07/09-11,
# with optional class_weight="balanced" via BCEWithLogitsLoss's
# pos_weight, computed from the actual inner-train split
# (evaluate.compute_pos_weight) rather than a fixed global constant.
def train_and_score_nested(train_uids, train_labels, train_family,
                            outer_uids, batch_size, lr, seed,
                            use_class_weight=False,
                            epochs=config.EPOCHS, patience=config.PATIENCE,
                            inner_splits=10):
    inner_train_idx, inner_val_idx = evaluate.make_folds(
        train_labels, train_family, n_splits=inner_splits, random_state=seed
    )[0]

    def subset(idxs):
        return ([train_uids[i] for i in idxs], [train_labels[i] for i in idxs])

    inner_train_uids, inner_train_labels = subset(inner_train_idx)
    inner_val_uids, inner_val_labels = subset(inner_val_idx)

    inner_train_ds = dataset.DatParkinsonDataset(inner_train_uids, inner_train_labels, load_fn=volume_cache.get)
    inner_val_ds = dataset.DatParkinsonDataset(inner_val_uids, inner_val_labels, load_fn=volume_cache.get)
    inner_train_loader = torch.utils.data.DataLoader(inner_train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
    inner_val_loader = torch.utils.data.DataLoader(inner_val_ds, batch_size=batch_size, num_workers=0)

    torch.manual_seed(seed)
    net = model.build_model().to(config.DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=config.WEIGHT_DECAY)
    if use_class_weight:
        pos_weight = evaluate.compute_pos_weight(inner_train_labels)
        loss_fn = torch.nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(pos_weight, device=config.DEVICE))
    else:
        pos_weight = None
        loss_fn = torch.nn.BCEWithLogitsLoss()

    best_state, history = train_mod.train_one_fold(
        net, inner_train_loader, inner_val_loader, optimizer, loss_fn,
        epochs=epochs, patience=patience, device=config.DEVICE,
        use_amp=config.USE_AMP, seed=seed,
    )
    net.load_state_dict(best_state)

    outer_ds = dataset.DatParkinsonDataset(outer_uids, load_fn=volume_cache.get)
    outer_loader = torch.utils.data.DataLoader(outer_ds, batch_size=batch_size, num_workers=0)
    outer_probs = []
    for x, _ in outer_loader:
        outer_probs.append(model.predict(net, x))
    return np.concatenate(outer_probs), history, best_state, pos_weight

In [ ]:
# [RUN ME] -- fold-0 sanity check. Confirms class_weight runs without
# error and reports a first, cheap outer-fold number (plus the computed
# pos_weight itself) before committing to the full 5x5 gate below.
batch_size, lr = 32, 2e-3  # rung 2/3's validated winner, held fixed here

outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                   n_splits=config.N_FOLDS, random_state=config.SEED)
fold0_train_idx, fold0_test_idx = outer_folds[0]
fold0_train_uids = [uids[i] for i in fold0_train_idx]
fold0_train_labels = [labels[i] for i in fold0_train_idx]
fold0_train_family = [families[i] for i in fold0_train_idx]
fold0_test_uids = [uids[i] for i in fold0_test_idx]
fold0_test_labels = np.array([labels[i] for i in fold0_test_idx])

start = time.time()
probs, history, best_state, pos_weight = train_and_score_nested(
    fold0_train_uids, fold0_train_labels, fold0_train_family,
    fold0_test_uids, batch_size=batch_size, lr=lr, seed=config.SEED,
    use_class_weight=True,
)
elapsed = time.time() - start
score = evaluate.log_loss_score(fold0_test_labels, probs)
print(f"fold 0 sanity check: pos_weight={pos_weight:.4f} (config.BASE_RATE-implied "
      f"global value would be {(1 - config.BASE_RATE) / config.BASE_RATE:.4f}), "
      f"{len(history['val_loss'])} epochs, inner-val best={min(history['val_loss']):.4f}, "
      f"outer log loss={score:.4f}, {elapsed:.1f}s ({elapsed / len(history['val_loss']):.2f}s/epoch)")
print("rung 3's fold-0/seed-42 log loss for comparison: 0.4005 (README.md)")

In [ ]:
# [RUN ME] -- full 5-fold nested CV, repeated 5x, with class_weight
# ("balanced" pos_weight). Same protocol as rung 3
# (notebooks/07_cnn_rung3.ipynb).
N_REPEATS = 5
oof_repeats_cw = []

for repeat_seed in range(config.SEED, config.SEED + N_REPEATS):
    outer_folds = evaluate.make_folds(np.array(labels), np.array(families),
                                       n_splits=config.N_FOLDS, random_state=repeat_seed)
    oof_probs = np.zeros(len(uids))
    for fold_i, (train_idx, test_idx) in enumerate(outer_folds):
        fold_train_uids = [uids[i] for i in train_idx]
        fold_train_labels = [labels[i] for i in train_idx]
        fold_train_family = [families[i] for i in train_idx]
        fold_test_uids = [uids[i] for i in test_idx]

        probs, history, best_state, pos_weight = train_and_score_nested(
            fold_train_uids, fold_train_labels, fold_train_family,
            fold_test_uids, batch_size=batch_size, lr=lr, seed=repeat_seed,
            use_class_weight=True,
        )
        oof_probs[test_idx] = probs
        torch.save(best_state, config.CHECKPOINT_DIR / f"rung4_classweight_seed{repeat_seed}_fold{fold_i}.pt")
        fold_score = evaluate.log_loss_score(np.array(labels)[test_idx], probs)
        print(f"  seed={repeat_seed} fold={fold_i}: pos_weight={pos_weight:.4f}, "
              f"{len(history['val_loss'])} epochs, outer fold log loss={fold_score:.4f}")

    repeat_logloss = evaluate.log_loss_score(np.array(labels), oof_probs)
    oof_repeats_cw.append(oof_probs)
    print(f"seed={repeat_seed} pooled OOF log loss: {repeat_logloss:.4f}")
    np.save(config.DATA_PROCESSED / f"rung4_classweight_oof_seed{repeat_seed}.npy", oof_probs)

repeat_scores_cw = np.array([evaluate.log_loss_score(np.array(labels), oof) for oof in oof_repeats_cw])
print(f"\n{N_REPEATS}-repeat class_weight CNN pooled log loss: "
      f"mean={repeat_scores_cw.mean():.4f}, sd={repeat_scores_cw.std(ddof=1):.4f}")
print("current validated CNN (rung 3, README.md 2026-09-09): mean=0.4520, sd=0.0109")

In [ ]:
# [RUN ME] -- paired bootstrap: class_weight CNN (this experiment's
# repeat 0, seed=42) vs. the current validated CNN (rung 3's repeat 0,
# seed=42, same split -- reloaded from disk, not retrained). Same
# discipline as notebooks 07, 09, 10, and 11's gate cells.
CURRENT_CNN_MEAN = 0.4520  # README.md 2026-09-09, rung 3, 5-repeat mean
CURRENT_CNN_SD = 0.0109    # same, ddof=1

y_true = np.array(labels)
current_cnn_oof = np.load(config.DATA_PROCESSED / "rung3_oof_seed42.npy")
cw_oof_for_pairing = oof_repeats_cw[0]

ci_low, ci_high = evaluate.paired_bootstrap_ci(
    y_true, cw_oof_for_pairing, current_cnn_oof, seed=config.SEED)
ci_favors_cw = ci_high < 0  # delta = class_weight - current; negative favors class_weight

repeat_mean_cw = repeat_scores_cw.mean()
repeat_sd_cw = repeat_scores_cw.std(ddof=1)
noise_threshold = 2 * max(repeat_sd_cw, CURRENT_CNN_SD)
beats_current_by = CURRENT_CNN_MEAN - repeat_mean_cw

gate_passed = ci_favors_cw and (beats_current_by > noise_threshold)

print(f"paired bootstrap delta (class_weight - current CNN), 95% CI: [{ci_low:+.4f}, {ci_high:+.4f}]")
print(f"5-repeat mean={repeat_mean_cw:.4f}, sd={repeat_sd_cw:.4f}; "
      f"beats current CNN ({CURRENT_CNN_MEAN}) by {beats_current_by:+.4f} "
      f"(2x max-sd noise threshold = {noise_threshold:.4f})")
print(f"GATE {'PASSED' if gate_passed else 'NOT PASSED'}: "
      f"{'class_weight REPLACES the current CNN.' if gate_passed else 'does not beat the current CNN by more than noise -- keep the current CNN.'}")

**What we're looking for:** does `class_weight="balanced"` (in place of
plain `BCEWithLogitsLoss` in rungs 0-3) beat the current validated CNN
(0.4520) by more than noise? Going in, expected to be a weak effect at
best (mild imbalance, and a specific prior argument against `pos_weight`
distorting log loss calibration — see intro cell).

**What we found:** *(paste: the computed pos_weight value; the fold-0
sanity check number; the 5-repeat mean/sd; the paired-bootstrap 95% CI;
the GATE PASSED/NOT PASSED line)*

**Decision / next step:** *(if the gate passed: surprising given the
prior argument against it -- double check the per-family/calibration
breakdown before trusting it, then this CNN replaces the current one in
the submission blend. If not, as expected: keep the current CNN, move
on to experiment 5 [NL-means denoising,
`notebooks/13_cnn_denoising.ipynb`] -- the last of the five, and the one
with the most real risk to the submission's 3-hour inference budget
since it touches `data.py`'s shared preprocessing. Log this as a
negative result either way per the project's standing rule.)*